In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import pandas as pd
import rivretrieve as rr

provider = rr.provider("cz_chmi")

## Provider info

In [ ]:
# Provider-level info: name, capabilities, API URLs
info = provider.info()
print(info)

# Parse the nested metadata JSON
meta = json.loads(info.metadata)
print(json.dumps(meta, indent=2))

In [ ]:
# All registered providers
rr.provider_info()

In [ ]:
# CHMI stations in the packaged catalogue (831 stations from the live metadata endpoint)
rr.provider("cz_chmi").stations()

In [ ]:
# All products across all providers
rr.products()

In [ ]:
# Map of Czech stations only
rr.map_stations(providers="cz_chmi")

In [ ]:
# CHMI discharge products
provider.products(observed_property="discharge")

In [ ]:
# CHMI stage products (daily mean + instantaneous)
provider.products(observed_property="stage")

In [ ]:
# CHMI water temperature products
provider.products(observed_property="water_temperature")

In [ ]:
# All products (5 total)
provider.products()

In [ ]:
# Station-product availability (all unknown — CHMI metadata doesn't expose per-variable availability)
provider.station_products()

In [ ]:
# Station-products for a single station
provider.station_products(stations=["0-203-1-200700"])

## Stations

In [ ]:
# All stations as a pandas DataFrame
stations_df = provider.stations().data.to_pandas()
stations_df

In [ ]:
# Expand the metadata JSON column into separate columns
pd.json_normalize(stations_df["metadata"].apply(json.loads))

In [ ]:
# Full station info for Praha-Výtoň on the Vltava (large drainage area, good for discharge)
station_id = "0-203-1-200700"
row = stations_df[stations_df["station_id"] == station_id].iloc[0]
print(row.to_string())
print()
print("metadata:")
print(json.dumps(json.loads(row["metadata"]), indent=2))

## Products

In [ ]:
# All CHMI products as a pandas DataFrame
products_df = provider.products().data.to_pandas()
products_df

In [ ]:
# Expand product metadata — shows tsConID, url_type, native_unit, unit_conversion
for _, row in products_df.iterrows():
    print(f"=== {row['product_id']} ===")
    print(json.dumps(json.loads(row["metadata"]), indent=2))
    print()

## Observations — daily mean discharge

CHMI timestamps carry a Z suffix (`2020-01-01T00:00:00Z`) — they are **unambiguously UTC**.
No timezone inference is needed. The series annotation `timezone_source` is `"provider_timestamp_utc"`.

In [ ]:
# Daily mean discharge for Praha-Výtoň on the Vltava (drainage area ~26 900 km²)
result = provider.observations(
    stations="0-203-1-200700",
    products="discharge_daily_mean",
    start=pd.Timestamp("2020-01-01"),
    end=pd.Timestamp("2022-12-31"),
)

df = result.data.to_pandas()
df

In [ ]:
# Time column is UTC-aware (Z suffix parsed directly, no conversion needed)
print("dtype:", df["time"].dtype)
print("first timestamp:", df["time"].iloc[0])
df.set_index("time")["value"].plot(
    title="Daily discharge — Praha-Výtoň, Vltava (m³/s)",
    figsize=(12, 4),
)

## Observations — daily mean stage (note: native cm → m conversion)

CHMI returns stage in **centimetres**. RivRetrieve converts to metres on ingest.
The raw cm value is preserved in the `raw_value` row annotation.

In [ ]:
result_stage = provider.observations(
    stations="0-203-1-200700",
    products="stage_daily_mean",
    start=pd.Timestamp("2020-01-01"),
    end=pd.Timestamp("2022-12-31"),
)
df_stage = result_stage.data.to_pandas()
df_stage

In [ ]:
df_stage.set_index("time")["value"].plot(
    title="Daily stage — Praha-Výtoň, Vltava (m, converted from native cm)",
    figsize=(12, 4),
)

## Observations — daily mean water temperature

In [ ]:
result_temp = provider.observations(
    stations="0-203-1-200700",
    products="water_temperature_daily_mean",
    start=pd.Timestamp("2021-01-01"),
    end=pd.Timestamp("2023-12-31"),
)
df_temp = result_temp.data.to_pandas()
df_temp

In [ ]:
df_temp.set_index("time")["value"].plot(
    title="Daily water temperature — Praha-Výtoň, Vltava (°C)",
    figsize=(12, 4),
)

## Observations — hourly instantaneous discharge

Hourly data is fetched from a separate HQ file (tsConID = QH).
The URL pattern is `H_{id}_HQ_{year}.json`.

In [ ]:
# Shorter window — hourly data is high-frequency
result_iv = provider.observations(
    stations="0-203-1-200700",
    products="discharge_instantaneous",
    start=pd.Timestamp("2020-03-01"),
    end=pd.Timestamp("2020-06-30"),
)
df_iv = result_iv.data.to_pandas()
print(f"{len(df_iv)} rows")
df_iv.head()

In [ ]:
df_iv.set_index("time")["value"].plot(
    title="Hourly instantaneous discharge — Praha-Výtoň, Vltava (m³/s)",
    figsize=(12, 4),
)

## Observations — hourly instantaneous stage

In [ ]:
result_stage_iv = provider.observations(
    stations="0-203-1-200700",
    products="stage_instantaneous",
    start=pd.Timestamp("2020-03-01"),
    end=pd.Timestamp("2020-06-30"),
)
df_stage_iv = result_stage_iv.data.to_pandas()
print(f"{len(df_stage_iv)} rows")
df_stage_iv.head()

In [ ]:
df_stage_iv.set_index("time")["value"].plot(
    title="Hourly instantaneous stage — Praha-Výtoň, Vltava (m, from native cm)",
    figsize=(12, 4),
)

## Provenance, annotations, and issues

In [ ]:
# Observation provenance — endpoints called, windows, request params
print(result.provenance)

In [ ]:
# Issues (missing data warnings, 404s for years with no data, etc.)
for issue in result.issues:
    print(issue)

In [ ]:
# Row-level annotations: tsConID, native unit (m3/s), converted unit, raw value
result.row_annotations.data.to_pandas()

In [ ]:
# Series-level annotations: timezone_source, resolved_timezone, url_type, endpoints, years
ann_df = result.series_annotations.data.to_pandas()
ann_df

In [ ]:
# timezone_source is 'provider_timestamp_utc' — Z suffix is authoritative, no inference
print(ann_df[ann_df["annotation"] == "timezone_source"][["annotation", "value"]].to_string())
print()
# url_type shows whether the daily (DQ) or hourly (HQ) file was used
print(ann_df[ann_df["annotation"] == "url_type"][["annotation", "value"]].to_string())
print()
# Returned time range
print(ann_df[ann_df["annotation"].isin(["returned_time_range_start", "returned_time_range_end"])][["annotation", "value"]].to_string())

In [ ]:
# Stage row annotations show raw_value in cm (native) alongside value in m (canonical)
row_ann_stage = result_stage.row_annotations.data.to_pandas()
print("Unique annotations:", row_ann_stage["annotation"].unique().tolist())
# Compare raw_value (cm) vs canonical value (m) for the same timestamp
raw = row_ann_stage[row_ann_stage["annotation"] == "raw_value"].head(3)
canonical = result_stage.data.to_pandas().head(3)
print("\nNative raw_value (cm):")
print(raw[["time", "value"]].to_string())
print("\nCanonical value (m):")
print(canonical[["time", "value"]].to_string())

## Bulk request — multiple stations and products

In [ ]:
# Two Vltava stations (Praha-Výtoň and Praha-Modřany), two products
result_bulk = provider.observations(
    stations=["0-203-1-200700", "0-203-1-200000"],
    products=["discharge_daily_mean", "stage_daily_mean"],
    start=pd.Timestamp("2022-01-01"),
    end=pd.Timestamp("2022-12-31"),
    on_issue="ignore",
)
bulk_df = result_bulk.data.to_pandas()
print(bulk_df[["station_id", "product_id"]].drop_duplicates())
bulk_df

In [ ]:
# Pivot to compare discharge at both stations
wide = bulk_df[bulk_df["product_id"] == "discharge_daily_mean"].pivot(
    index="time", columns="station_id", values="value"
)
wide.plot(title="Daily discharge — Praha-Výtoň vs Praha-Modřany, Vltava (m³/s)", figsize=(12, 4))

In [ ]:
# Using the top-level rr.observations() wrapper (delegates to provider handle)
result2 = rr.observations(
    provider="cz_chmi",
    stations="0-203-1-200700",
    products="discharge_daily_mean",
    start=pd.Timestamp("2023-01-01"),
    end=pd.Timestamp("2023-12-31"),
    on_issue="ignore",
)
result2.data.to_pandas().set_index("time")["value"].plot(
    title="Daily discharge — Praha-Výtoň, Vltava (m³/s) via rr.observations()"
)

## Map — Czech stations only

In [ ]:
# Filter map to Czech Republic stations only
rr.map_stations(providers="cz_chmi")

In [ ]:
# Filter by bounding box: Bohemia roughly (12.0, 49.0, 16.5, 51.2)
rr.map_stations(providers="cz_chmi", bbox=(12.0, 49.0, 16.5, 51.2))

## Annotation schemas (declared by the provider)

In [ ]:
# Row annotation schema — what annotation names the provider emits per observation
for s in provider.row_annotation_schema():
    print(f"{s.annotation_id:30s} [{s.value_type}]  {s.description}")

In [ ]:
# Series annotation schema — what annotation names the provider emits per station-product series
for s in provider.series_annotation_schema():
    print(f"{s.annotation_id:30s} [{s.value_type}]  {s.description}")